In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import load_iris, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("iris-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

/home/madhav/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://localhost:5000


In [2]:
SEED = 0
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
y = y.astype(int)
X = X / 255.0
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)

def train_and_log(hidden_layer_sizes, learning_rate_init, batch_size, epochs, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("Hidden Layer Sizes", hidden_layer_sizes)
        mlflow.log_param("Learning Rate Init", learning_rate_init)
        mlflow.log_param("Batch Size", batch_size)
        mlflow.log_param("Epochs", epochs)

        model = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, learning_rate_init=learning_rate_init, batch_size=batch_size, max_iter=1, random_state=SEED)
        classes = np.arange(10)

        for epoch in range(epochs):
            model.partial_fit(X_train, y_train, classes=classes)
            val_preds = model.predict(X_val)
            mlflow.log_metric("train_loss", model.loss_, step=epoch)
            mlflow.log_metric("val_accuracy", accuracy_score(y_val, val_preds), step=epoch)

        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(model, name="model", skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"])

        run_id = run.info.run_id
        print(f"Logged run {run_id} | acc={acc:.4f} | f1={f1:.4f}")
        return run_id

In [3]:
run1 = train_and_log((128,), 0.1, 128, 25, run_name="mlp-1")
run2 = train_and_log((128,), 0.01, 128, 25, run_name="mlp-2")
run3 = train_and_log((128,), 0.001, 128, 25, run_name="mlp-3")

run4 = train_and_log((32,), 0.01, 128, 25, run_name="mlp-4")
run5 = train_and_log((64,), 0.01, 128, 25, run_name="mlp-5")
run6 = train_and_log((256,), 0.01, 128, 25, run_name="mlp-6")

Logged run 63c7edc8e0a54e8ea03b7024582a8a10 | acc=0.7594 | f1=0.7541
🏃 View run mlp-1 at: http://localhost:5000/#/experiments/1/runs/63c7edc8e0a54e8ea03b7024582a8a10
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 64d4f98b21c242f884a0c58b2012d5f3 | acc=0.9731 | f1=0.9730
🏃 View run mlp-2 at: http://localhost:5000/#/experiments/1/runs/64d4f98b21c242f884a0c58b2012d5f3
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 92ead480c6ff4b139e0c556af1412c00 | acc=0.9769 | f1=0.9768
🏃 View run mlp-3 at: http://localhost:5000/#/experiments/1/runs/92ead480c6ff4b139e0c556af1412c00
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 0b14d301cd5c439e83aad33abc7c9a1e | acc=0.9589 | f1=0.9585
🏃 View run mlp-4 at: http://localhost:5000/#/experiments/1/runs/0b14d301cd5c439e83aad33abc7c9a1e
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 25c94e3dde6e4d7880664852835a2509 | acc=0.9640 | f1=0.9638
🏃 View run mlp-5 at: http:/